# DOCX → Azota — lưu vào Drive folder **markdown azota**

Mọi code + `azota_out` + zip nằm ở:
`/content/drive/MyDrive/markdown azota`

Không dùng `/content/docx-to-azota` (mất khi tắt runtime).

**Xóa runtime cũ:** Runtime → Disconnect and delete runtime. Upload **chỉ** notebook này.

1. **Phần A** — mount Drive → extract → zip **trong folder của bạn**
2. **Phần B** — chỉ khi cần `$latex$` từ MathType

Cấm: `unimernet[full]` / `pip install tokenizers` / `transformers==4.42.4`.
Đừng bấm Stop lúc clone. Không Run all.


## Phần A — extract Azota


### A1. GPU (không bắt buộc cho extract)


In [ ]:
!nvidia-smi -L || echo CPU


### A2. Mount Drive + đưa code vào **markdown azota**

Ô này hỏi quyền Google Drive. Chọn đúng Google account có folder `markdown azota`.
Nếu folder chưa có, ô sẽ **tạo** `MyDrive/markdown azota`.


In [ ]:
import shutil, sys, subprocess
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

REPO = "https://github.com/phuchoang2603/refurbished-marketplace.git"
BRANCH = "cursor/docx-to-azota-pipeline-4d56"
STAGING = Path("/content/_repo_azota")

def _norm(name):
    return " ".join(name.strip().lower().replace("_", " ").replace("-", " ").split())

def find_markdown_azota():
    roots = [Path("/content/drive/MyDrive"), Path("/content/drive/My Drive")]
    hits = []
    for root in roots:
        if not root.is_dir():
            continue
        for child in root.iterdir():
            if child.is_dir() and _norm(child.name) == "markdown azota":
                hits.append(child)
            if not child.is_dir():
                continue
            try:
                for grand in child.iterdir():
                    if grand.is_dir() and _norm(grand.name) == "markdown azota":
                        hits.append(grand)
            except OSError:
                pass
    if hits:
        exact = [p for p in hits if p.name == "markdown azota"]
        return exact[0] if exact else hits[0]
    dest = Path("/content/drive/MyDrive/markdown azota")
    dest.mkdir(parents=True, exist_ok=True)
    return dest

ROOT = find_markdown_azota()
SKIP = {"azota_out", "azota_out.zip", "models", "uploads", "__pycache__", ".pytest_cache", ".git", ".ipynb_checkpoints"}

if STAGING.exists():
    shutil.rmtree(STAGING)
subprocess.check_call([
    "git", "clone", "-b", BRANCH, "--depth", "1", "--single-branch",
    "--filter=blob:none", "--sparse", REPO, str(STAGING),
])
subprocess.check_call(["git", "-C", str(STAGING), "sparse-checkout", "set", "tools/docx-to-azota"])
src = STAGING / "tools/docx-to-azota"
if not (src / "convert.py").exists():
    raise SystemExit("clone chưa đủ file — chạy lại ô này, đừng bấm Stop")

for item in src.iterdir():
    if item.name in SKIP:
        continue
    target = ROOT / item.name
    if item.is_dir():
        shutil.copytree(item, target, dirs_exist_ok=True)
    else:
        shutil.copy2(item, target)

for sub in ("uploads", "azota_out", "models"):
    (ROOT / sub).mkdir(exist_ok=True)

p = str(ROOT)
if p in sys.path:
    sys.path.remove(p)
sys.path.insert(0, p)
print("ROOT", ROOT)
print("OK", (ROOT / "convert.py").exists())


### A3. Import converter (không load UniMERNet)


In [ ]:
!pip -q install pillow
from convert import convert_docx
from eval_timer import StepTimer
timer = StepTimer()
OUT = str(ROOT / "azota_out")
print("import OK")
print("OUT", OUT)


### A4. Upload `.docx` (lưu vào Drive `uploads/`)


In [ ]:
from google.colab import files
uploaded = files.upload()
if uploaded:
    name = next(iter(uploaded))
    dest = ROOT / "uploads" / name
    dest.write_bytes(uploaded[name])
    DOCX = str(dest)
else:
    DOCX = str(ROOT / "samples" / "de-vat-li-lan-3.docx")
print(DOCX)


### A5. Extract — **bắt buộc**. Kỳ vọng mathml 69, mathtype 16, img 8.


In [ ]:
from pathlib import Path
with timer.step("Bước 1", "OOXML"):
    man = convert_docx(DOCX, OUT)
print(man["counts"])
print("\n".join(Path(OUT, "markup.txt").read_text(encoding="utf-8").splitlines()[:25]))


### A6. Zip **trong folder Drive** — có thể dừng tại đây


In [ ]:
import shutil
from google.colab import files
zip_base = ROOT / "azota_out"
archive = shutil.make_archive(str(ROOT / "azota_out"), "zip", root_dir=str(ROOT), base_dir="azota_out")
print("đã lưu Drive", archive)
files.download(archive)
print("xong phần A")


## Phần B — tùy chọn: MathType → LaTeX

Chỉ chạy nếu Azota cần `$latex$` thay `[!m:$mathtype_N$]`. T4. Không OCR hình.
Checkpoint UniMERNet lưu tại `markdown azota/models` (lần sau không tải lại).


### B1. ImageMagick


In [ ]:
!apt-get -qq install -y imagemagick libmagickwand-dev
!pip -q install Wand huggingface_hub
print("ImageMagick OK")


### B2. UniMERNet `--no-deps` (vá onnx + pytorch_utils)


In [ ]:
from install_colab import allow_wmf_in_imagemagick, install_unimernet_colab
allow_wmf_in_imagemagick()
install_unimernet_colab()


### B3. Raster WMF + nhận dạng + gắn `$latex$`


In [ ]:
from pathlib import Path
import shutil
from colab_opt import prepare_unimernet_checkpoint, free_cuda, vision_jobs_from_manifest, inject_latex_into_markup
from vision import rasterize_formula_image, load_unimernet, unimernet_batch
from convert import apply_unimernet_latex
from google.colab import files

png_dir = Path(OUT) / "sidecar_png"
png_dir.mkdir(exist_ok=True)
jobs = []
with timer.step("Bước 2", "raster WMF"):
    for aid, src in vision_jobs_from_manifest(man, OUT, kinds=("mathtype",)):
        dest = png_dir / f"{aid}.png"
        got = rasterize_formula_image(src, dest, dpi=200)
        if got:
            jobs.append((aid, got))
print(len(jobs), "ảnh công thức")

models_dir = str(ROOT / "models")
with timer.step("Bước 3-load", "tiny"):
    cfg = prepare_unimernet_checkpoint("tiny", models_dir)
    model, vis, device = load_unimernet(cfg_path=cfg, fp16=True)
print("device", device)

with timer.step("Bước 3", "batch"):
    preds = unimernet_batch(model, vis, device, jobs, batch_size=8)
apply_unimernet_latex(man, preds, Path(OUT))
p = Path(OUT) / "markup.txt"
text = inject_latex_into_markup(p.read_text(encoding="utf-8"), preds)
p.write_text(text, encoding="utf-8")
timer.print_summary()
for k, v in list(preds.items())[:5]:
    print(k, "→", v[:80])
free_cuda(model, vis)

archive = shutil.make_archive(str(ROOT / "azota_out"), "zip", root_dir=str(ROOT), base_dir="azota_out")
print("đã lưu Drive", archive)
files.download(archive)
